<a href="https://colab.research.google.com/github/pajola/secai-course/blob/main/module-02-data-security/04_data_risk_report.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" width="150"></a>

# 4 · The Data Risk Report — one audit, one decision, one sign-off

**SecAI course · Data Security for AI, part 4**

Notebooks 1–3 each looked at one risk. A reviewer has to give one answer: **can this dataset be used for training, yes or no, and under which conditions?**
This notebook puts every check into a single, repeatable `audit()`, produces the report, applies the remediations, and runs the audit again.
It follows the six steps of the review in order, so it can also be used on its own as a short live demo.

| Step | Check | From |
|---|---|---|
| 1 | Inventory | notebook 1 |
| 2 | PII, including free text | notebook 1 |
| 3 | Classification and re-identification risk | notebook 1 |
| 4 | Class balance and representation | notebook 2 |
| 5 | Bias audit (disparate impact) | notebook 2 |
| 6 | Provenance and integrity | notebook 3 |

The checks are rewritten here in compact form so that the notebook runs on its own.

In [1]:
# Setup — on Google Colab, fetch the dataset generator and install Faker (all other packages are preinstalled).
import os, sys, urllib.request
if "google.colab" in sys.modules:
    if not os.path.exists("northvale.py"):
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/pajola/secai-course/main/module-02-data-security/northvale.py", "northvale.py")
    %pip install -q faker

In [2]:
import re, numpy as np, pandas as pd
from IPython.display import Markdown, display
from northvale import load, batch_sha256, reexport_from_source, REF_DATE

df, manifest = load()
print(f"{len(df)} rows x {df.shape[1]} columns:", ", ".join(df.columns))

1200 rows x 13 columns: applicant_id, full_name, email, phone, date_of_birth, gender, postcode, annual_income_eur, credit_score, loan_amount_eur, advisor_notes, source_batch, approved


## The audit

Each check returns one finding: what was measured, the threshold, a status, and the action if it fails.
Statuses: **FAIL** blocks training; **WARN** must be documented; **CONDITION** must be enforced later in the pipeline; **PASS**.

In [3]:
DIRECT_IDENTIFIERS = ["full_name", "email", "phone", "date_of_birth"]
IBAN_RE = re.compile(r"\bIT\d{2}(?: ?[A-Z0-9]){23}\b")
HEALTH_RE = r"hospital\w*|sick leave|chemotherapy|diabetic|pregnan\w*|medication|depression|surgery"
IDDOC_RE = r"ID card|passport|driving licence"
QUASI_IDENTIFIERS = ["date_of_birth", "age_band", "postcode", "gender"]

def iban_valid(s):
    s = s.replace(" ", ""); s = s[4:] + s[:4]
    return int("".join(str(int(c, 36)) for c in s)) % 97 == 1

def age_band(dob, bands=(0, 30, 40, 50, 60, 200), labels=("<30", "30-39", "40-49", "50-59", "60+")):
    age = (REF_DATE - pd.to_datetime(dob)).dt.days // 365.25
    return pd.cut(age, list(bands), right=False, labels=list(labels)).astype(str)

def finding(step, check, measured, threshold, status, action=""):
    return dict(step=step, check=check, measured=measured, threshold=threshold, status=status, action=action)

def audit(data, manifest, audit_table, fairness_gate=None):
    """data: the training candidate; audit_table: applicant_id, gender, age_band (kept apart, for steps 4-5)."""
    out, n = [], len(data)
    groups = audit_table.set_index("applicant_id").loc[data.applicant_id].reset_index()

    # 1-2 · PII
    present = [c for c in DIRECT_IDENTIFIERS if c in data]
    out.append(finding(2, "direct identifiers", ", ".join(present) or "none", "none",
                       "FAIL" if present else "PASS", "drop name/email/phone; DOB → age band"))
    if "advisor_notes" in data:
        notes = data.advisor_notes
        iban = notes.str.findall(IBAN_RE).apply(lambda f: any(iban_valid(x) for x in f)).sum()
        health, iddoc = notes.str.contains(HEALTH_RE, case=False).sum(), notes.str.contains(IDDOC_RE, case=False).sum()
        out.append(finding(2, "PII in free text", f"IBAN {iban} · health {health} · ID docs {iddoc} rows", "0 rows",
                           "FAIL" if iban + health + iddoc else "PASS", "drop or redact advisor_notes"))
    else:
        out.append(finding(2, "PII in free text", "no free-text fields", "0 rows", "PASS"))

    # 3 · re-identification
    qi = [c for c in QUASI_IDENTIFIERS if c in data]
    key = data[qi].astype(str).agg("|".join, axis=1)
    k = key.map(key.value_counts()).min()
    out.append(finding(3, "k-anonymity", f"k = {k} on {', '.join(qi)}", "k ≥ 5",
                       "FAIL" if k == 1 else "WARN" if k < 5 else "PASS", "generalise quasi-identifiers"))

    # 4 · balance and representation
    minority = data.approved.value_counts(normalize=True).min()
    out.append(finding(4, "label balance", f"minority class {minority:.1%}", "≥ 30%", "PASS" if minority >= 0.3 else "WARN"))
    share = groups.gender.value_counts(normalize=True)
    out.append(finding(4, "gender representation", f"M {share.get('M', 0):.1%} / F {share.get('F', 0):.1%}", "each ≥ 40%",
                       "WARN" if share.min() < 0.4 else "PASS", "document; monitor per-group performance"))
    smallest = groups.age_band.value_counts(normalize=True)
    out.append(finding(4, "age representation", f"{smallest.idxmin()}: {smallest.min():.1%} ({(groups.age_band == smallest.idxmin()).sum()} rows)",
                       "each band ≥ 5%", "WARN" if smallest.min() < 0.05 else "PASS", "model card: known limitation; human review"))

    # 5 · bias
    rate = data.approved.groupby(groups.gender.values).mean()
    di = rate["F"] / rate["M"]
    status = "PASS" if di >= 0.8 else "CONDITION" if fairness_gate else "FAIL"
    out.append(finding(5, "disparate impact (labels)", f"DI F/M = {di:.2f}", "≥ 0.80", status,
                       "investigate labels and postcode proxy; fairness gate DI ≥ 0.80 on the model before promotion"))

    # 6 · provenance and integrity
    m = manifest.set_index("batch_id")
    batches = data.source_batch.unique()
    no_basis = [b for b in batches if not m.legal_basis.get(b)]
    out.append(finding(6, "legal basis", f"{', '.join(no_basis) or 'all batches documented'}", "every batch",
                       "FAIL" if no_basis else "PASS", "exclude until a legal basis and license are documented"))
    bad = [b for b in batches if m.sha256.get(b) and batch_sha256(data[data.source_batch == b].reindex(columns=df.columns)) != m.sha256[b]]
    missing = [b for b in batches if not m.sha256.get(b)]
    out.append(finding(6, "integrity (SHA-256)", "; ".join([f"{b}: MISMATCH" for b in bad] + [f"{b}: no checksum" for b in missing]) or "all match",
                       "all match", "FAIL" if bad or missing else "PASS", "quarantine, re-export from source, re-verify"))
    return pd.DataFrame(out)

One detail in the integrity check: the hash is computed on the batch **as it left the source**, with all its columns. A remediated dataset has fewer columns
(identifiers dropped), so the pipeline verifies each batch **before** transforming it. The `verified_batches` list below records that.

## Run 1 — the dataset as delivered

In [4]:
audit_table = pd.DataFrame({"applicant_id": df.applicant_id, "gender": df.gender, "age_band": age_band(df.date_of_birth)})
report = audit(df, manifest, audit_table)

def show(report):
    icon = {"FAIL": "🔴 FAIL", "WARN": "🟡 WARN", "CONDITION": "🔵 CONDITION", "PASS": "🟢 PASS"}
    decision = ("**BLOCKED** for training until remediated" if (report.status == "FAIL").any() else
                "**APPROVED WITH CONDITIONS**" if report.status.isin(["WARN", "CONDITION"]).any() else "**APPROVED**")
    rows = "\n".join(f"| {r.step} | {r.check} | {r.measured} | {r.threshold} | {icon[r.status]} |" for r in report.itertuples())
    display(Markdown(f"### Decision: {decision}\n\n| Step | Check | Measured | Threshold | Status |\n|---|---|---|---|---|\n{rows}"))

show(report)

### Decision: **BLOCKED** for training until remediated

| Step | Check | Measured | Threshold | Status |
|---|---|---|---|---|
| 2 | direct identifiers | full_name, email, phone, date_of_birth | none | 🔴 FAIL |
| 2 | PII in free text | IBAN 110 · health 97 · ID docs 86 rows | 0 rows | 🔴 FAIL |
| 3 | k-anonymity | k = 1 on date_of_birth, postcode, gender | k ≥ 5 | 🔴 FAIL |
| 4 | label balance | minority class 41.2% | ≥ 30% | 🟢 PASS |
| 4 | gender representation | M 69.3% / F 30.7% | each ≥ 40% | 🟡 WARN |
| 4 | age representation | 60+: 3.7% (44 rows) | each band ≥ 5% | 🟡 WARN |
| 5 | disparate impact (labels) | DI F/M = 0.72 | ≥ 0.80 | 🔴 FAIL |
| 6 | legal basis | B3_broker_purchase | every batch | 🔴 FAIL |
| 6 | integrity (SHA-256) | B2_core_banking: MISMATCH; B3_broker_purchase: no checksum | all match | 🔴 FAIL |

Six findings block training, matching notebooks 1–3. The report also lists what each finding requires:

In [5]:
def actions(report):
    todo = report[report.status != "PASS"]
    display(Markdown("\n".join(f"{i}. **{r.check}** ({r.status}): {r.action}" for i, r in enumerate(todo.itertuples(), 1))))
actions(report)

1. **direct identifiers** (FAIL): drop name/email/phone; DOB → age band
2. **PII in free text** (FAIL): drop or redact advisor_notes
3. **k-anonymity** (FAIL): generalise quasi-identifiers
4. **gender representation** (WARN): document; monitor per-group performance
5. **age representation** (WARN): model card: known limitation; human review
6. **disparate impact (labels)** (FAIL): investigate labels and postcode proxy; fairness gate DI ≥ 0.80 on the model before promotion
7. **legal basis** (FAIL): exclude until a legal basis and license are documented
8. **integrity (SHA-256)** (FAIL): quarantine, re-export from source, re-verify

## Remediation

1. **PII:** drop direct identifiers and `advisor_notes`; date of birth → age band. Gender leaves the features and moves to the access-controlled audit table.
2. **Legal:** exclude B3 until a legal basis and license are documented.
3. **Integrity:** quarantine B2, re-export it from the source, and accept it only if its hash matches the manifest.
4. **Re-identification:** merge the two oldest age bands into "50+" in the training data, so that no age band + postcode group is tiny.
5. **Fairness:** the labels can't be fixed by a transformation. Accept the dataset with a **fairness gate**: the trained model must reach DI ≥ 0.80 on a held-out set before promotion (notebook 2, section 2.5).
6. **Representation:** document the 60+ under-coverage in the model card.

In [6]:
def verified_source(batch_id, manifest):
    """Re-export a batch and accept it only if it matches the manifest checksum."""
    batch = reexport_from_source(batch_id)
    assert batch_sha256(batch) == manifest.set_index("batch_id").sha256[batch_id], f"{batch_id}: still does not verify"
    return batch

verified_batches = {b: verified_source(b, manifest) for b in ["B1_core_banking", "B2_core_banking"]}   # B3 excluded (no legal basis)
raw = pd.concat(verified_batches.values(), ignore_index=True)

audit_table = pd.DataFrame({"applicant_id": raw.applicant_id, "gender": raw.gender, "age_band": age_band(raw.date_of_birth)})
remediated = raw.drop(columns=["full_name", "email", "phone", "advisor_notes", "gender"]).assign(
    age_band=age_band(raw.date_of_birth, bands=(0, 30, 40, 50, 200), labels=("<30", "30-39", "40-49", "50+"))).drop(columns="date_of_birth")
print(f"{len(remediated)} rows x {remediated.shape[1]} columns:", ", ".join(remediated.columns))

942 rows x 8 columns: applicant_id, postcode, annual_income_eur, credit_score, loan_amount_eur, source_batch, approved, age_band


## Run 2 — after remediation

The integrity check in `audit()` compares each batch with its manifest hash, which needs the full original columns: we pass it the verified raw batches
for that check, and the remediated table for everything else.

In [7]:
def audit_remediated(remediated, raw, manifest, audit_table, fairness_gate):
    report = audit(remediated, manifest, audit_table, fairness_gate)
    integrity = audit(raw, manifest, audit_table).query("check == 'integrity (SHA-256)'")      # verified before transformation
    report.loc[report.check == "integrity (SHA-256)"] = integrity.values
    return report

report2 = audit_remediated(remediated, raw, manifest, audit_table, fairness_gate=0.8)
show(report2)
actions(report2)

### Decision: **APPROVED WITH CONDITIONS**

| Step | Check | Measured | Threshold | Status |
|---|---|---|---|---|
| 2 | direct identifiers | none | none | 🟢 PASS |
| 2 | PII in free text | no free-text fields | 0 rows | 🟢 PASS |
| 3 | k-anonymity | k = 13 on age_band, postcode | k ≥ 5 | 🟢 PASS |
| 4 | label balance | minority class 45.3% | ≥ 30% | 🟢 PASS |
| 4 | gender representation | M 69.4% / F 30.6% | each ≥ 40% | 🟡 WARN |
| 4 | age representation | 60+: 3.7% (35 rows) | each band ≥ 5% | 🟡 WARN |
| 5 | disparate impact (labels) | DI F/M = 0.74 | ≥ 0.80 | 🔵 CONDITION |
| 6 | legal basis | all batches documented | every batch | 🟢 PASS |
| 6 | integrity (SHA-256) | all match | all match | 🟢 PASS |

1. **gender representation** (WARN): document; monitor per-group performance
2. **age representation** (WARN): model card: known limitation; human review
3. **disparate impact (labels)** (CONDITION): investigate labels and postcode proxy; fairness gate DI ≥ 0.80 on the model before promotion

Nothing blocks training any more. What is left are not data problems that a transformation can fix:

* **Representation** (gender 70/30, 60+ at about 4%) can only be fixed by collecting data; until then it is documented.
* **The labels still carry historical bias** (DI ≈ 0.7). The dataset is accepted on the condition that the *model* passes the fairness gate.
* k-anonymity passes (k = 13) only because the training data keeps just age band and postcode. Adding any other quasi-identifier later means running the audit again.

## The report

What goes to the data owner, the AI security lead and the DPO for sign-off:

In [8]:
def risk_report(before, after, n_before, n_after):
    lines = [
        "# Data Risk Report — Northvale loan-approval dataset",
        f"**Scope:** {n_before} applications, 3 ingestion batches · **Reviewed:** {pd.Timestamp('today').date()}",
        f"\n**Initial decision:** BLOCKED ({(before.status == 'FAIL').sum()} blocking findings)",
        f"\n**Decision after remediation:** APPROVED WITH CONDITIONS · {n_after} rows (B1 + verified B2), {len(remediated.columns)} columns",
        "\n## Remediation applied",
        "1. **PII:** direct identifiers and `advisor_notes` dropped; date of birth → age band; gender moved to the audit table.",
        "2. **Legal:** B3 (broker purchase, 258 rows) excluded until a legal basis and license are documented.",
        "3. **Integrity:** B2 quarantined, re-exported from the core banking system, hash verified. 40 labels had been flipped (rejected → approved).",
        "4. **Re-identification:** age bands 50-59 and 60+ merged in the training data.",
        "\n## Conditions",
        "* **Fairness gate:** the model must reach DI F/M ≥ 0.80 on a held-out set before promotion. Investigate the labels and the postcode proxy.",
        "* **Model card, known limitations:** " + "; ".join(after.query("status == 'WARN'").apply(lambda r: f"{r.check}: {r.measured}", axis=1)),
        "\n## Sign-off",
        "| Role | Name | Decision | Date |\n|---|---|---|---|",
        "| Data owner (Credit Risk Office) | | | |\n| AI security | | | |\n| Data Protection Officer | | | |",
    ]
    return "\n".join(lines)

text = risk_report(report, report2, len(df), len(remediated))
display(Markdown(text))
with open("data_risk_report.md", "w") as f: f.write(text)

# Data Risk Report — Northvale loan-approval dataset
**Scope:** 1200 applications, 3 ingestion batches · **Reviewed:** 2026-11-19

**Initial decision:** BLOCKED (6 blocking findings)

**Decision after remediation:** APPROVED WITH CONDITIONS · 942 rows (B1 + verified B2), 8 columns

## Remediation applied
1. **PII:** direct identifiers and `advisor_notes` dropped; date of birth → age band; gender moved to the audit table.
2. **Legal:** B3 (broker purchase, 258 rows) excluded until a legal basis and license are documented.
3. **Integrity:** B2 quarantined, re-exported from the core banking system, hash verified. 40 labels had been flipped (rejected → approved).
4. **Re-identification:** age bands 50-59 and 60+ merged in the training data.

## Conditions
* **Fairness gate:** the model must reach DI F/M ≥ 0.80 on a held-out set before promotion. Investigate the labels and the postcode proxy.
* **Model card, known limitations:** gender representation: M 69.4% / F 30.6%; age representation: 60+: 3.7% (35 rows)

## Sign-off
| Role | Name | Decision | Date |
|---|---|---|---|
| Data owner (Credit Risk Office) | | | |
| AI security | | | |
| Data Protection Officer | | | |

The report is also saved as `data_risk_report.md`, ready to attach to the training request. Because `audit()` is code, the review can run **every time the data changes**,
for example as a CI step before each training run, instead of once.

## Optional — what a generic profiler finds

[ydata-profiling](https://github.com/ydataai/ydata-profiling) generates a full statistical profile in one line: types, missing values, distributions, correlations, duplicates.
It is a good first step for the inventory (step 1). Set `RUN_PROFILE = True` to try it (it installs a few extra packages and takes about a minute).

In [9]:
RUN_PROFILE = False

if RUN_PROFILE:
    if "google.colab" in sys.modules:
        %pip install -q ydata-profiling "setuptools<81"
    from ydata_profiling import ProfileReport
    ProfileReport(df, title="Northvale loans — as delivered", minimal=True).to_file("northvale_profile.html")
    print("written: northvale_profile.html")

Compare the profile with the audit above. The profiler flags the high-cardinality columns (names, emails) and shows the distributions, but it has no concept of
**an IBAN inside a note**, a **legal basis**, a **checksum**, or **disparate impact**. A profiler describes the data; a security review asks what the data
exposes, who is harmed by it, and whether it can be trusted.

## Take-aways

* *A dataset is an attack surface, a liability, and a decision record, all at once. Review it before it becomes a model.*
* One review, six steps, one decision: **BLOCKED** until remediated, then **APPROVED WITH CONDITIONS**.
* Remediation changes the data (minimise, exclude, re-export); conditions move the remaining risk to a later gate (fairness on the model) or to documentation (model card).
* Write the review as code: it becomes repeatable, testable, and can run before every training job.
* The decision needs **three signatures**: the data owner, AI security and the DPO. None of them can sign off alone.

### References
* Gebru et al., *Datasheets for Datasets*, CACM 2021 · Mitchell et al., *Model Cards for Model Reporting*, FAT* 2019.
* NIST AI Risk Management Framework (AI RMF 1.0), 2023: *Map* and *Measure* functions.
* ISO/IEC 42001:2023, *AI management systems* · EU AI Act, Art. 10 (data and data governance).
* GDPR Art. 35 (data protection impact assessment) and Art. 37–39 (Data Protection Officer).